Prueba

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, LSTM, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [2]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\Desktop\\pruebas_collab\\datosNarmax\\24pasos_lstm_consumption.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [3]:
datos.head()

,temp,zone1,zone2,zone3,hour,e
date,,,,,,
2017-01-01 00:00:00,-2.051356,0.151445,-0.851669,0.033500,-1.660858,NaN
2017-01-01 00:10:00,-2.074813,-0.433079,-0.211097,0.016502,-1.660858,NaN
2017-01-01 00:20:00,-2.091152,-0.527709,-0.283791,-0.055068,-1.660858,NaN
2017-01-01 00:30:00,-2.122213,-0.651648,-0.411186,-0.174054,-1.660858,NaN
2017-01-01 00:40:00,-2.154568,-0.774750,-0.507632,-0.244729,-1.660858,NaN


In [4]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Espacio de búsqueda

In [5]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas LSTM
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades LSTM
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'epochs': hp.choice('epochs', [2 ** i for i in range(3, 9)]),  # Número de épocas de entrenamiento
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes LSTM, es decir [observaciones, retardos, caracteristicas]

In [6]:
futuros = 24
pasados  = 12

In [7]:
datosX = []
datosY = []
for i in range(pasados, len(datos) - futuros + 1):
  datosX.append(datos.iloc[i-pasados:i, 0:datos.shape[1]])
  datosY.append(datos.iloc[i+futuros-1:i+futuros, 1])


In [8]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (52381, 12, 6)
Dimensiones de Y: (52381, 1)


In [9]:
print(datosX[0])

[[-2.05135573  0.15144479 -0.85166941  0.03349964 -1.66085756         nan]
 [-2.07481311 -0.43307937 -0.21109699  0.01650173 -1.66085756         nan]
 [-2.0911524  -0.52770864 -0.28379117 -0.05506842 -1.66085756         nan]
 [-2.12221321 -0.65164785 -0.41118591 -0.17405378 -1.66085756         nan]
 [-2.15456823 -0.77474965 -0.50763164 -0.2447293  -1.66085756         nan]
 [-2.16556893 -0.87272862 -0.59759968 -0.29303915 -1.66085756         nan]
 [-2.19986525 -0.95898362 -0.68109001 -0.32166721 -1.51634012         nan]
 [-2.22332263 -1.03518949 -0.74658674 -0.39681586 -1.51634012         nan]
 [-2.19387957 -1.12730648 -0.83223632 -0.46391287 -1.51634012         nan]
 [-2.22413151 -1.19597551 -0.88909611 -0.49969795 -1.51634012         nan]
 [-2.22008713 -1.24873342 -0.98842083 -0.52385287 -1.51634012         nan]
 [-2.22736701 -1.29730419 -1.03232523 -0.5614272  -1.51634012         nan]]


Se dividen los conjuntos de datos

In [10]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (36666, 12, 6)
Las dimensiones de testX son:  (10529, 12, 6)
Las dimensiones de valX son:  (5186, 12, 6)


In [11]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (36666, 1)
Las dimensiones de testY son:  (10529, 1)
Las dimensiones de valY son:  (5186, 1)


Se crean métricas para medir desempeño

In [12]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

Versión Final


In [13]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(testX.shape[1], testX.shape[2])))
    if (params['layers'] == 1):
      model.add(LSTM(units=params['units'], activation=params['activation'], return_sequences=False))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(LSTM(units=params['units'], activation=params['activation'], return_sequences=True))
          model.add(Dropout(params['dropout']))
      model.add(LSTM(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(testX, testY, epochs=params['epochs'],
                        validation_split=0.3,
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])


    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [14]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=50, trials=trials, rstate=np.random.default_rng(42))

print(best)

  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

231/231 - 14s - 59ms/step - ia: 0.3067 - loss: 0.6730 - mae: 0.6578 - rmse: 0.8128 - smape: 1.4584 - val_ia: 0.2699 - val_loss: 0.6183 - val_mae: 0.6382 - val_rmse: 0.6969 - val_smape: 1.2163

Epoch 2/128                                           

231/231 - 4s - 17ms/step - ia: 0.5890 - loss: 0.4006 - mae: 0.4916 - rmse: 0.6262 - smape: 0.9305 - val_ia: 0.3173 - val_loss: 0.3666 - val_mae: 0.5040 - val_rmse: 0.5546 - val_smape: 0.9713

Epoch 3/128                                           

231/231 - 4s - 19ms/step - ia: 0.6781 - loss: 0.2957 - mae: 0.4201 - rmse: 0.5389 - smape: 0.7693 - val_ia: 0.3458 - val_loss: 0.2957 - val_mae: 0.4549 - val_rmse: 0.5023 - val_smape: 0.8840

Epoch 4/128                                           

231/231 - 4s - 15ms/step - ia: 0.7074 - loss: 0.2560 - mae: 0.3920 - rmse: 0.5006 - smape: 0.7236 - val_ia: 0.3592 - val_loss: 0.2691 - val_mae: 0.4345 - val_rmse: 0.4805 - val_smape: 0.8484

Epoch 5

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/16                                                                          

29/29 - 16s - 543ms/step - ia: 0.1535 - loss: 0.7303 - mae: 0.6993 - rmse: 0.8516 - smape: 1.6988 - val_ia: 0.4219 - val_loss: 0.4513 - val_mae: 0.5639 - val_rmse: 0.6574 - val_smape: 1.0483

Epoch 2/16                                                                          

29/29 - 4s - 146ms/step - ia: 0.7112 - loss: 0.2656 - mae: 0.3843 - rmse: 0.5077 - smape: 0.7119 - val_ia: 0.6738 - val_loss: 0.2028 - val_mae: 0.3678 - val_rmse: 0.4414 - val_smape: 0.7371

Epoch 3/16                                                                          

29/29 - 4s - 148ms/step - ia: 0.8053 - loss: 0.1423 - mae: 0.2839 - rmse: 0.3757 - smape: 0.5494 - val_ia: 0.7321 - val_loss: 0.1562 - val_mae: 0.3066 - val_rmse: 0.3892 - val_smape: 0.6567

Epoch 4/16                                                                          

29/29 - 4s - 148ms/step - ia: 0.8433 - loss: 0.0920 - mae: 0.2308 - rmse: 0.3022 -

...

922/922 - 13s - 14ms/step - ia: 0.7512 - loss: 0.1719 - mae: 0.3009 - rmse: 0.3725 - smape: 0.6323 - val_ia: 0.2801 - val_loss: 0.1549 - val_mae: 0.2934 - val_rmse: 0.3105 - val_smape: 0.6214

Epoch 2/8                                                                            

922/922 - 7s - 7ms/step - ia: 0.8478 - loss: 0.0635 - mae: 0.1946 - rmse: 0.2416 - smape: 0.4711 - val_ia: 0.2689 - val_loss: 0.1375 - val_mae: 0.2911 - val_rmse: 0.3084 - val_smape: 0.6246

Epoch 3/8                                                                            

922/922 - 7s - 7ms/step - ia: 0.8590 - loss: 0.0545 - mae: 0.1797 - rmse: 0.2236 - smape: 0.4516 - val_ia: 0.2984 - val_loss: 0.1087 - val_mae: 0.2506 - val_rmse: 0.2676 - val_smape: 0.5518

Epoch 4/8                                                                            

922/922 - 6s - 6ms/step - ia: 0.8700 - loss: 0.0462 - mae: 0.1653 - rmse: 0.2063 - smape: 0.4278 - val_ia: 0.3084 - val_loss: 0.0913 - val_mae: 0.2317 - val_rmse: 0

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



58/58 - 8s - 135ms/step - ia: 0.1331 - loss: 0.7693 - mae: 0.7246 - rmse: 0.8755 - smape: 1.6989 - val_ia: 0.3332 - val_loss: 0.8362 - val_mae: 0.7629 - val_rmse: 0.9041 - val_smape: 1.5449

Epoch 2/64                                                                           

58/58 - 1s - 14ms/step - ia: 0.1598 - loss: 0.7397 - mae: 0.7104 - rmse: 0.8591 - smape: 1.6311 - val_ia: 0.3432 - val_loss: 0.8132 - val_mae: 0.7505 - val_rmse: 0.8911 - val_smape: 1.5077

Epoch 3/64                                                                           

58/58 - 1s - 22ms/step - ia: 0.1989 - loss: 0.7068 - mae: 0.6930 - rmse: 0.8396 - smape: 1.5429 - val_ia: 0.3531 - val_loss: 0.7801 - val_mae: 0.7335 - val_rmse: 0.8724 - val_smape: 1.4519

Epoch 4/64                                                                           

58/58 - 1s - 23ms/step - ia: 0.2468 - loss: 0.6646 - mae: 0.6710 - rmse: 0.8146 - smape: 1.4591 - val_ia: 0.3629 - val_loss: 0.7359 - val_mae: 0.7112 - val_rmse: 0.8469

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                          

461/461 - 14s - 30ms/step - ia: 0.4306 - loss: 0.5587 - mae: 0.5891 - rmse: 0.7164 - smape: 1.2102 - val_ia: 0.2149 - val_loss: 0.4112 - val_mae: 0.5416 - val_rmse: 0.5764 - val_smape: 0.8312

Epoch 2/128                                                                          

461/461 - 6s - 13ms/step - ia: 0.7146 - loss: 0.2460 - mae: 0.3885 - rmse: 0.4879 - smape: 0.6731 - val_ia: 0.2381 - val_loss: 0.4112 - val_mae: 0.5247 - val_rmse: 0.5564 - val_smape: 0.8040

Epoch 3/128                                                                          

461/461 - 6s - 13ms/step - ia: 0.7388 - loss: 0.2023 - mae: 0.3536 - rmse: 0.4418 - smape: 0.6479 - val_ia: 0.2416 - val_loss: 0.4233 - val_mae: 0.5330 - val_rmse: 0.5652 - val_smape: 0.7961

Epoch 4/128                                                                          

461/461 - 6s - 14ms/step - ia: 0.7667 - loss: 0.1657 - mae: 0.3185 - rmse: 

In [15]:
print(best)

{'activation': 0, 'batch': 1, 'dropout': 0.1, 'epochs': 4, 'layers': 3.0, 'learning_rate': 0.002646330986430552, 'units': 0}
